In [6]:
#imports 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import f_classif
from sklearn.impute import KNNImputer

In [7]:
df=pd.read_csv("cust_segmentation_Data.csv")
print(df.index)
print("All columns are")
ai=0
for i in df:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=850, step=1)
All columns are
0 Customer Id
1 Age
2 Edu
3 Years Employed
4 Income
5 Card Debt
6 Other Debt
7 Defaulted
8 DebtIncomeRatio


In [10]:
df.info()
print()
print('First let us fill the empty values')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer Id      850 non-null    int64  
 1   Age              850 non-null    int64  
 2   Edu              850 non-null    int64  
 3   Years Employed   850 non-null    int64  
 4   Income           850 non-null    int64  
 5   Card Debt        850 non-null    float64
 6   Other Debt       850 non-null    float64
 7   Defaulted        850 non-null    int64  
 8   DebtIncomeRatio  850 non-null    float64
dtypes: float64(3), int64(6)
memory usage: 59.9 KB

First let us fill the empty values


In [25]:
imputer = KNNImputer(n_neighbors=5)

df["Defaulted"] = imputer.fit_transform(df[["Age", "Edu", "Years Employed", "Income", 
                                            "Card Debt", "Other Debt", "DebtIncomeRatio", "Defaulted"]])[:, -1]

df["Defaulted"] = df["Defaulted"].round().astype(int)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer Id      850 non-null    int64  
 1   Age              850 non-null    int64  
 2   Edu              850 non-null    int64  
 3   Years Employed   850 non-null    int64  
 4   Income           850 non-null    int64  
 5   Card Debt        850 non-null    float64
 6   Other Debt       850 non-null    float64
 7   Defaulted        850 non-null    int64  
 8   DebtIncomeRatio  850 non-null    float64
dtypes: float64(3), int64(6)
memory usage: 59.9 KB


In [12]:
# As the data has already been encoded now forst let's test without scaling
tx = df.drop(['Defaulted'],axis=1)
ty = df.iloc[:,7]
print(tx)
print(ty)

     Customer Id  Age  Edu  Years Employed  Income  Card Debt  Other Debt  \
0              1   41    2               6      19      0.124       1.073   
1              2   47    1              26     100      4.582       8.218   
2              3   33    2              10      57      6.111       5.802   
3              4   29    2               4      19      0.681       0.516   
4              5   47    1              31     253      9.308       8.908   
..           ...  ...  ...             ...     ...        ...         ...   
845          846   27    1               5      26      0.548       1.220   
846          847   28    2               7      34      0.359       2.021   
847          848   25    4               0      18      2.802       3.210   
848          849   32    1              12      28      0.116       0.696   
849          850   52    1              16      64      1.866       3.638   

     DebtIncomeRatio  
0                6.3  
1               12.8  
2     

In [15]:
tX_train, tX_test, ty_train, ty_test = train_test_split( tx, ty,test_size=0.2)

tmodel = LogisticRegression(max_iter=1000)

tmodel.fit(tX_train, ty_train)


ty_pred = tmodel.predict(tX_test)


print("Accuracy:", accuracy_score(ty_test, ty_pred))

print("\nClassification Report:")
print(classification_report(ty_test, ty_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(ty_test, ty_pred))

Accuracy: 0.8176470588235294

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.92      0.88       128
           1       0.68      0.50      0.58        42

    accuracy                           0.82       170
   macro avg       0.76      0.71      0.73       170
weighted avg       0.81      0.82      0.81       170


Confusion Matrix:
[[118  10]
 [ 21  21]]


In [16]:
tscores = cross_val_score(tmodel, tx, ty, cv=10, scoring="accuracy")

print('Actual cross-val results')
print(f"Scores for each fold: {tscores}")
print(f"Mean Accuracy: {tscores.mean():.2f}")
print(f"Standard Deviation: {tscores.std():.2f}")

Actual cross-val results
Scores for each fold: [0.82352941 0.8        0.84705882 0.8        0.75294118 0.82352941
 0.78823529 0.78823529 0.84705882 0.84705882]
Mean Accuracy: 0.81
Standard Deviation: 0.03


In [25]:
print('''That was a pretty good score 
Now let's scale the data and drop useless columns
''')

That was a pretty terrible score 
Now let's scale the data and drop useless columns



In [17]:
fx = df.drop(columns=["Defaulted"])
fy = df["Defaulted"]

In [18]:
fscores, pvalues = f_classif(fx, fy)

anovadf = pd.DataFrame({"Feature": fx.columns,"F-Score": fscores,"p-value": pvalues})

anovadf = anovadf.sort_values(by="F-Score",ascending=False)

print(anovadf.round(4))

           Feature   F-Score  p-value
7  DebtIncomeRatio  156.2408   0.0000
3   Years Employed   79.1381   0.0000
5        Card Debt   45.6396   0.0000
1              Age   25.2394   0.0000
6       Other Debt   12.6301   0.0004
2              Edu    9.2899   0.0024
4           Income    7.8562   0.0052
0      Customer Id    0.0012   0.9724


In [19]:
print('We can safely drop Edu Income and Customer Id')

We can safely drop Edu Income and Customer Id


In [20]:
df=pd.read_csv("cust_segmentation_Data.csv")
print(df.index)
print("All columns are")
ai=0
for i in df:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=850, step=1)
All columns are
0 Customer Id
1 Age
2 Edu
3 Years Employed
4 Income
5 Card Debt
6 Other Debt
7 Defaulted
8 DebtIncomeRatio


In [21]:
aadf = df.drop(['Customer Id','Income','Edu'],axis=1)
print(aadf.columns)

Index(['Age', 'Years Employed', 'Card Debt', 'Other Debt', 'Defaulted',
       'DebtIncomeRatio'],
      dtype='object')


In [26]:
X = aadf.drop('Defaulted',axis=1)
y = df['Defaulted']
print(X.columns)
print(y)

Index(['Age', 'Years Employed', 'Card Debt', 'Other Debt', 'DebtIncomeRatio'], dtype='object')
0      0
1      0
2      1
3      0
4      0
      ..
845    0
846    0
847    1
848    0
849    0
Name: Defaulted, Length: 850, dtype: int64


In [43]:
# First let's test using standard scaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,)


preprocessor = ColumnTransformer([
    ('standard', StandardScaler(), ['Age', 'Years Employed', 'Card Debt', 'Other Debt', 'DebtIncomeRatio']),
], remainder='passthrough')


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

In [44]:
X_train_transformed = pipeline.named_steps['preprocessor'].transform(X_train)

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

transformed_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

print(transformed_df.head())

pt = pipeline.named_steps['preprocessor'].named_transformers_['standard']


   standard__Age  standard__Years Employed  standard__Card Debt  \
0       0.102734                  0.497407             0.066453   
1      -0.762777                 -0.378854            -0.567611   
2       1.339177                  1.957840             0.818702   
3       0.720956                 -0.963027            -0.687671   
4       0.350023                  1.811797             1.409151   

   standard__Other Debt  standard__DebtIncomeRatio  
0             -0.053543                  -0.360757  
1             -0.331869                   0.162488  
2              2.703975                   0.104350  
3             -0.374491                  -0.535172  
4              0.509117                   0.889218  


In [45]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")

print(classification_report(y_test, y_pred))

Accuracy: 0.788235294117647

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.94      0.87       127
           1       0.67      0.33      0.44        43

    accuracy                           0.79       170
   macro avg       0.74      0.64      0.65       170
weighted avg       0.77      0.79      0.76       170



In [46]:
# Checking through pipeline
scores = cross_val_score(pipeline ,X ,y ,cv=10,scoring='accuracy')

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Cross Validation Scores: [0.82352941 0.81176471 0.84705882 0.82352941 0.76470588 0.82352941
 0.81176471 0.77647059 0.84705882 0.87058824]
Mean Accuracy: 0.82
Standard Deviation: 0.030246965016899868


In [47]:
print('Not much improvement but now it takes less number of iterations <maybe?>')

Not much improvement but now it takes less number of iterations <maybe?>


In [48]:
# Now let's check using Yeo-Johnson

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,)


preprocessor = ColumnTransformer([
    ('yeo_johnson', PowerTransformer(method='yeo-johnson', standardize=True), ['Age', 'Years Employed', 'Card Debt',
                                                                               'Other Debt', 'DebtIncomeRatio']),
], remainder='passthrough')


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs'))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

In [49]:
X_train_transformed = pipeline.named_steps['preprocessor'].transform(X_train)

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

transformed_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

print(transformed_df.head())


pt = pipeline.named_steps['preprocessor'].named_transformers_['yeo_johnson']
print()
print(pt.lambdas_)

   yeo_johnson__Age  yeo_johnson__Years Employed  yeo_johnson__Card Debt  \
0          0.332736                    -0.524878               -0.066992   
1         -0.577855                    -0.524878               -0.609404   
2         -1.675072                    -0.745155               -0.519017   
3         -0.039191                     0.286727               -1.029458   
4         -1.852562                    -1.774185               -1.249272   

   yeo_johnson__Other Debt  yeo_johnson__DebtIncomeRatio  
0                -0.027441                      0.528869  
1                -0.884670                     -0.928507  
2                -0.806355                     -0.444399  
3                -1.978186                     -1.757867  
4                -0.855602                     -0.592458  

[ 0.18436117  0.36731577 -0.82068433 -0.40665587  0.24340956]


In [50]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7705882352941177

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.93      0.86       124
           1       0.64      0.35      0.45        46

    accuracy                           0.77       170
   macro avg       0.72      0.64      0.65       170
weighted avg       0.75      0.77      0.75       170


Confusion Matrix:
[[115   9]
 [ 30  16]]


In [53]:
# Checking through pipeline
scores = cross_val_score(pipeline ,X ,y ,cv=10,scoring='accuracy')

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Cross Validation Scores: [0.84705882 0.8        0.83529412 0.81176471 0.75294118 0.77647059
 0.8        0.77647059 0.8        0.84705882]
Mean Accuracy: 0.8047058823529412
Standard Deviation: 0.029855476565763574


In [52]:
fscores, pvalues = f_classif(X, y)

anovadf = pd.DataFrame({"Feature": X.columns,"F-Score": fscores,"p-value": pvalues})

anovadf = anovadf.sort_values(by="F-Score",ascending=False)

print(anovadf.round(4))

           Feature   F-Score  p-value
4  DebtIncomeRatio  156.2408   0.0000
1   Years Employed   79.1381   0.0000
2        Card Debt   45.6396   0.0000
0              Age   25.2394   0.0000
3       Other Debt   12.6301   0.0004


In [57]:
# Now let's check using Yeo-Johnson without dropping any column
X = df.drop(['Defaulted','Customer Id'],axis=1)
y = df['Defaulted']
print(X.columns)
print(y)

Index(['Age', 'Edu', 'Years Employed', 'Income', 'Card Debt', 'Other Debt',
       'DebtIncomeRatio'],
      dtype='object')
0      0
1      0
2      1
3      0
4      0
      ..
845    0
846    0
847    1
848    0
849    0
Name: Defaulted, Length: 850, dtype: int64


In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,)


preprocessor = ColumnTransformer([
    ('yeo_johnson', PowerTransformer(method='yeo-johnson', standardize=True), ['Age', 'Years Employed', 'Card Debt',
                                                                               'Other Debt', 'DebtIncomeRatio']),
], remainder='passthrough')


pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs',max_iter=1000))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

In [63]:
X_train_transformed = pipeline.named_steps['preprocessor'].transform(X_train)

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

transformed_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

print(transformed_df.head())


pt = pipeline.named_steps['preprocessor'].named_transformers_['yeo_johnson']
print()
print(pt.lambdas_)

   yeo_johnson__Age  yeo_johnson__Years Employed  yeo_johnson__Card Debt  \
0         -0.255325                     0.908348               -1.071710   
1         -0.255325                    -0.326347               -0.948406   
2          0.135173                    -1.341736               -0.444956   
3          1.435626                     0.016519               -1.118210   
4          0.609777                     1.112527                0.454684   

   yeo_johnson__Other Debt  yeo_johnson__DebtIncomeRatio  remainder__Edu  \
0                -1.271562                     -1.514898             1.0   
1                -1.112549                     -0.865755             3.0   
2                 0.207354                      0.545856             2.0   
3                -1.972756                     -2.213937             2.0   
4                 1.426656                     -0.349329             3.0   

   remainder__Income  
0               36.0  
1               23.0  
2               2

In [64]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8176470588235294

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       128
           1       0.74      0.40      0.52        42

    accuracy                           0.82       170
   macro avg       0.78      0.68      0.71       170
weighted avg       0.81      0.82      0.80       170


Confusion Matrix:
[[122   6]
 [ 25  17]]


In [66]:
# Checking through pipeline
scores = cross_val_score(pipeline ,X ,y ,cv=10,scoring='accuracy')

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

Cross Validation Scores: [0.83529412 0.77647059 0.84705882 0.77647059 0.75294118 0.77647059
 0.82352941 0.78823529 0.83529412 0.85882353]
Mean Accuracy: 0.8070588235294117
Standard Deviation: 0.03497898528780824
